In [2506]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (make_scorer,roc_auc_score,roc_curve, precision_score, fbeta_score, f1_score, recall_score,
confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore") # suppress noncritical warnings to keep output clean

# Read Heart Failure Dataset

In [2508]:
df = pd.read_csv("..\data\heart_failure_clinical_records_dataset.csv")
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


We prepare dataset for machine learning modeling

# Define X and y

In [2511]:
drop_cols = ["time", "DEATH_EVENT"] # we droped time to avoid data leakage because follow up time is part of the survival outcomes
X = df.drop(columns = drop_cols)
y = df["DEATH_EVENT"]
X.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0


In [2512]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: DEATH_EVENT, dtype: int64

# Split Dataset

In [2514]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [2515]:
# We extract numeric and categorical features
numeric_cols = ["age", "creatinine_phosphokinase", "ejection_fraction", "platelets", "serum_creatinine", "serum_sodium"]
categorical_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex","smoking"]

# Logistic Regression Model

In [2517]:
#We preprocess the columns in a transformer
preprocessor = ColumnTransformer(transformers =[("num", StandardScaler(), numeric_cols),("cat", "passthrough", categorical_cols)], remainder="drop")

In [2518]:
# We buid a pipeline
log_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                        ("classifier",LogisticRegression(max_iter=1000, solver="lbfgs", C = 0.1, class_weight="balanced", random_state= 42))])

In [2519]:
# We fit the model
log_pipeline.fit(X_train, y_train);

We check the model performance on training set

In [2521]:
y_train_pred = log_pipeline.predict(X_train)
print(classification_report(y_train, y_train_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.88      0.78      0.83       162
   death_event       0.63      0.77      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.78      0.76       239
  weighted avg       0.80      0.78      0.78       239



We check the model performance on test set

In [2523]:
y_test_pred = log_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



In [2524]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[32  9]
 [ 6 13]]


The logistic regression model correctly identified 68% of the actual death-event cases in the test set. There were 19 actual death-event cases, and the model detected approximately 13 of them. The precision for the death-event class was also 0.59, meaning that among patients predicted as death-event cases, about 59% truly experienced a death event. Although the model shows moderate ability to detect mortality cases, it still misses some of the actual death-event patients, which is an important limitation in a healthcare risk prediction setting.

## Hyperparameter Tuning

We tune the hyperparameters and employ a cross validation for possible improve performance.

In [2528]:
para_grid = {"classifier__C": [0.01, 0.02, 0.03,0.04,0.05], "classifier__penalty": ["l2"], 
             "classifier__solver": ["lbfgs"], "classifier__class_weight": ["balanced"] }

In [2529]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state =42)
score = make_scorer(recall_score,labels =[1], average="macro")
random_search = RandomizedSearchCV(estimator=log_pipeline,
                          param_distributions= para_grid,
                          n_iter = 5,
                          cv = cv,
                          scoring=score,
                          refit = True,
                          random_state=42,
                          n_jobs=-1)

In [2530]:
random_search.fit(X_train, y_train);

In [2531]:
print("Best parameters:", random_search.best_params_)

Best parameters: {'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__class_weight': 'balanced', 'classifier__C': 0.03}


In [2532]:
# We choose the best model
best_logistic_model = random_search.best_estimator_

In [2533]:
# We predict using the best model
y_train_pred = best_logistic_model.predict(X_train)

In [2534]:
# We check performance on training set
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.80      0.83       162
   death_event       0.63      0.74      0.68        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



In [2535]:
# We check performance on test set
y_test_pred = best_logistic_model.predict(X_test)

In [2536]:
# We check the performance
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.86      0.78      0.82        41
   death_event       0.61      0.74      0.67        19

      accuracy                           0.77        60
     macro avg       0.74      0.76      0.74        60
  weighted avg       0.78      0.77      0.77        60



After applying cross-validation and hyperparameter tuning, the logistic regression model achieved a recall of 0.74 for the death-event class on the test set. This means the model correctly detected approximately 74% of the patients who actually experienced a death event. The model also achieved a precision of 0.61 for the death-event class, meaning that among patients predicted as death-event cases, 61% were truly death-event patients. Overall, the tuned model shows moderate ability to identify mortality-risk cases, with improved sensitivity to death events, although some false positives and false negatives remain. The similarity between the training and test performance suggests that the model generalizes reasonably well and does not show strong evidence of overfitting.

# Random Forest Model

Next we fit the RF model.

In [2540]:
# This part is not really neccessary since tree based models are not sensitive to scaling, but I've reatined it since I'm learning to maintain a professional workflow.

rf_preprocessor = ColumnTransformer(transformers = [("num", "passthrough",numeric_cols ), ("cat","passthrough", categorical_cols)], remainder = "drop")

In [2541]:
# We build a RF model pipeline
rf_pipeline = Pipeline(steps = [("preprocessor", rf_preprocessor), ("classifier",RandomForestClassifier(n_estimators = 50,
                                                                                                       max_depth=10,
                                                                                                       min_samples_leaf=10,
                                                                                                       
                                                                                                       class_weight= "balanced",
                                                                                                       n_jobs= -1,
                                                                                                       random_state= 42))])


In [2542]:
# We fit RF model
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

Next we evaluate the performance of the RF model on the training set

In [2544]:
y_train_pred = rf_pipeline.predict(X_train)

We evaluate the performance of the RF model

In [2546]:
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.89      0.86      0.87       162
   death_event       0.72      0.78      0.75        77

      accuracy                           0.83       239
     macro avg       0.81      0.82      0.81       239
  weighted avg       0.84      0.83      0.83       239



We evaluate the performance on the test set

In [2548]:
y_test_pred = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.76      0.79        41
   death_event       0.57      0.68      0.62        19

      accuracy                           0.73        60
     macro avg       0.70      0.72      0.71        60
  weighted avg       0.75      0.73      0.74        60



## Hyperparameter Tuning

We tune the parameters and use cross validation for possible model improvement

In [2551]:
# Choice of hyperparameter

# several possible params combinations have been used. This choice seems better among others tested.
param_grid_rf = {"classifier__n_estimators":[3], 
                 "classifier__max_depth": [10],
                 "classifier__min_samples_leaf": [3,5,10,15],
                
                "classifier__class_weight": ["balanced"]}

In [2552]:
# Cross validation step
rf_cv = StratifiedKFold(n_splits = 5, shuffle=True,random_state=42)

In [2553]:
# We prioritize class 1 recall
score = make_scorer(recall_score, pos_label = 1)


In [2554]:
# We search for the best parameters
grid_search_rf = GridSearchCV(estimator = rf_pipeline,
                                   param_grid = param_grid_rf,
                                   cv= rf_cv,
                                   scoring=score,
                                   refit=True,
                                   n_jobs=-1)

In [2555]:
# We proceed to fit the model
grid_search_rf.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__class_weight': ['balanced'], 'classifier__max_depth': [10], 'classifier__min_samples_leaf': [3, 5, ...], 'classifier__n_estimators': [3]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(r..., pos_label=1)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: 

In [2556]:
# We check the best parameters
print("Best parameters:", grid_search_rf.best_params_)

Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 3}


We choose the best model

In [2558]:
best_RF_model = grid_search_rf.best_estimator_

We predict using the best model

In [2560]:
y_train_pred = best_RF_model.predict(X_train)

We check the model performance on the training set

In [2562]:
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.79      0.83       162
   death_event       0.63      0.75      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



We check the performance on test set

In [2564]:
y_test_pred = best_RF_model.predict(X_test)


In [2565]:
print(classification_report(y_test, y_test_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



Interpretation: